# Line follower project
## Robot Control Applications
#### Group Biddiscombe, Martinez, Spanò

In [19]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Sequence, Tuple, List, Dict
import numpy as np
import roboticstoolbox as rtb
import spatialgeometry as sg
import spatialmath as sm
import swift
import time
import math
import matplotlib.pyplot as plt
import matplotlib as mtb

In [20]:
# from unicycle_controller import TrajectoryTracker


In [21]:
# Set default matplotlib params
mtb.rcParams.update(mtb.rcParamsDefault)

# Global default values
FLOOR_HEIGHT = 0.02
WORLD_BOUNDS = [0., 10., 0., 10.]

COLOUR_BLUE = (0.2, 0.5, 0.8, 1.0)   # Robot colour
COLOUR_WALL = (1.0, 0.2, 0.2, 1.0)   # Obstacle colour
# COLOUR_RAY = (1.0, 0.8, 0.0, 0.7)    # Sensor ray colour (Yellow)
COLOUR_GREEN = (0.2, 0.8, 0.3, 0.7)  # Waypoint/Target colour
COLOUR_PATH = (0.1, 0.6, 0.3, 0.8)   # Path line colour
COLOUR_WHITE = (1.0, 1.0, 1.0, 0.6)  # World ground

In [22]:
# SETUP FUNCTIONS
# functions for the environment creation and setup

def add_floor(e: swift.Swift, world_bounds = WORLD_BOUNDS, floor_height: float = FLOOR_HEIGHT) -> sg.Cuboid:
    min_x, max_x, min_y, max_y = world_bounds
    width, depth = max_x - min_x, max_y - min_y
    floor = sg.Cuboid(scale=[width, depth, FLOOR_HEIGHT], color=COLOUR_WHITE)
    floor.T = sm.SE3((min_x + max_x) / 2.0, (min_y + max_y) / 2.0, floor_height)
    e.add(floor)
    return floor

def create_path(world_bounds = WORLD_BOUNDS, type: str = "line") -> List[Tuple[float, float]]: 
    min_x, max_x, min_y, max_y = world_bounds

    if type == "line":
        # Random line: y = a x + b, stay in positive area
        a = np.random.uniform(0.5, 2.)
        b = 0 # np.random.uniform(0., 2.)

        max_x_line = min(max_x, (max_y - b) / a)
        num_waypoints = 50
        xs = np.linspace(min_x, max_x_line, num_waypoints)
        waypoints = [(float(x), float(a * x + b)) for x in xs]

        print(f"[ENV] Path constant line:    y = {a:.3f} x + {b:.3f}")
    
    if type == "sine":
        a = np.random.uniform(0.5, 2.)
        b = (min_y + max_y) / 2 # always centred
        c = np.random.uniform(-1, 1)

        num_waypoints = 70
        xs = xs = np.linspace(min_x, max_x, num_waypoints)
        waypoints = [(float(x), float(np.sin(x * a + c) + b)) for x in xs]
        print(f"[ENV] Path sine wave:    y = sin({a:.3f} x + {c:.3f}) + {b:.3f}")

    print(f"[ENV] Path start state:  x = {waypoints[0][0]}, y = {waypoints[0][1]}")
    print(f"[ENV] Path end state:    x = {waypoints[-1][0]}, y = {waypoints[-1][1]}")
    return waypoints

def add_path_visual(e: swift.Swift, waypoints: List[Tuple[float, float]], floor_height: float = FLOOR_HEIGHT) -> None:
    """Draws grey static lines connecting the waypoints."""
    for i in range(len(waypoints) - 1):
        p1, p2 = waypoints[i], waypoints[i+1]
        dx, dy = p2[0] - p1[0], p2[1] - p1[1]
        length = np.hypot(dx, dy)
        angle = np.arctan2(dy, dx)
        cx, cy = (p1[0] + p2[0]) / 2.0, (p1[1] + p2[1]) / 2.0
        
        line = sg.Cuboid(scale=[length, 0.03, 0.01], color=COLOUR_PATH)
        line.T = sm.SE3(cx, cy, floor_height + 0.01) * sm.SE3.Rz(angle)
        e.add(line)

In [23]:
def get_random_coords_in_bounds(world_bounds = WORLD_BOUNDS, floor_height: float = FLOOR_HEIGHT) -> Tuple[float, float, float]:
    x = np.random.uniform(world_bounds[0] + 1, world_bounds[1] - 1)
    y = np.random.uniform(world_bounds[2] + 1, world_bounds[3] - 1)
    return (x, y, floor_height)

def get_position_on_line(waypoints: List[Tuple[float, float]], noise: float = 0.5, padding: float = 0.2, floor_height = FLOOR_HEIGHT) -> Tuple[float, float, float]:
    # ignore first 20% and last 20% of line (default padding is 0.2)
    pos_on_line = np.random.randint(int(padding * len(waypoints)), int((1 - padding) * len(waypoints)))
    pos_on_line = waypoints[pos_on_line]
    x_noise = np.random.uniform(-noise, noise)
    y_noise = np.random.uniform(-noise, noise)
    return (pos_on_line[0] + x_noise, pos_on_line[1] + y_noise, floor_height)
    
def add_block(e: swift.Swift, cx: float, cy: float, w: float, l: float, floor_height: float = FLOOR_HEIGHT) -> sg.Cuboid:
    """Helper function to add a static block to the environment."""
    block = sg.Cuboid(scale=[w, l, 0.5], color=COLOUR_WALL)
    block.T = sm.SE3(cx, cy, floor_height + 0.25)
    e.add(block)
    return block

def add_cylinder(e: swift.Swift, cx: float, cy: float, r: float, floor_height: float = FLOOR_HEIGHT) -> sg.Cylinder:
    """Helper function to add a static cylinder to the environment."""
    cylinder = sg.Cylinder(radius=r, length=0.5, color=COLOUR_WALL)
    cylinder.T = sm.SE3(cx, cy, floor_height + 0.25)
    e.add(cylinder)
    return cylinder

def create_obstacles(e: swift.Swift, waypoints, num_obstacles: int = 6, mixed_objects: bool = False, spacing: float = 1., world_bounds = WORLD_BOUNDS):
    current_positions = []
    obstacles = []

    while len(current_positions) < num_obstacles:
        prob = np.random.uniform(0,1)
        if prob > 0.8:
            pos = get_random_coords_in_bounds()
        elif prob > 0.5:
            pos = get_position_on_line(waypoints, noise = 1.5)
        else:
            pos = get_position_on_line(waypoints, noise = 0.5)
        
        for item in current_positions:
            d = np.hypot(item[0] - pos[0], item[1] - pos[1])
            if d < spacing: 
                # space them a little apart, if too close discard and repick a location
                continue
        current_positions.append(pos)
        
        if mixed_objects:
            prob = np.random.uniform(0,1)
            if prob > 0.5:
                radius = float(np.random.uniform(0.4, 0.7))
                obstacles.append(add_cylinder(e, pos[0], pos[1], radius))
                continue # skips the rest if a cylinder has been added, else adds a block

        width = float(np.random.uniform(0.4, 2.))
        length = float(np.random.uniform(0.4, 2.))

        obstacles.append(add_block(e, pos[0], pos[1], width, length))

    print(f"[ENV] Created {len(obstacles)} obstacles.")
    return obstacles


In [24]:
# Simulation timing settings
dt = 0.05
simulation_time = 40.0
simulation_steps = int(simulation_time / dt)

In [25]:
# Launch Swift
env = swift.Swift()
env.launch(realtime=True, comms="rtc", browser="notebook")

# time.sleep(1.0)


In [27]:
# Set the camera pose
env.set_camera_pose(position=[5.5, 1.0, 7.0], look_at=[5.5, 3., 2.0])
env.step(0)

In [28]:
# Add the world visuals:

# Add the floor, defines world borders
FLOOR = add_floor(env)

# Create the path, can be of type "line" for a straight path or "sine" for a curved path
WAYPOINTS = create_path(type="sine") # "line" or "sine"

# Add the path visually to the enviroment
add_path_visual(env, WAYPOINTS)

# Add obstacles, can be just blocks or of mixed block and cylinder type
walls = create_obstacles(env, waypoints=WAYPOINTS, mixed_objects=True, num_obstacles=5)

# Lock all the changes into the environment by performing a step
env.step(0)


[ENV] Path sine wave:    y = sin(0.569 x + 0.416) + 5.000
[ENV] Path start state:  x = 0.0, y = 5.404540961261983
[ENV] Path end state:    x = 10.0, y = 4.827810016777535
[ENV] Created 5 obstacles.


In [29]:
# Place the robot into the setup

# start_state = [0.0, -4.0, np.pi/2] # Start at (0, -4), facing UP (pi/2)

# # Place robot at start of line
# start_x, start_y = WAYPOINTS[0]



In [30]:

# Data Logging setup
time_log, x_log, y_log = [], [], []
v_cmd_log, w_cmd_log = [], []
tracking_error_log = []

target_arrived = False
